# 06. 접근성 지표별 이용건수 상관 비교

- 목적: 접근성 지표별로 서울 구·중분류 이용건수와의 상관관계를 비교함.
- 단위: `시군구 × 중분류`
- 기준연도: 2025년 이용건수
- 비교지표: 정부식 접근성, 선호 미반영 SFCA, 선호 반영 H3SFCA


## 1. 분석 경로 설정

- 기존 access 산출물을 그대로 사용함.
- 비교 결과만 `OUTPUT/compare_index`에 저장함.
- 원자료 복제 파일은 생성하지 않음.


In [ ]:
import pathlib
import numpy as np
import pandas as pd

BASE_PATH = pathlib.Path().resolve()

if BASE_PATH.name == "access":
    BASE_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    BASE_PATH = BASE_PATH.parent
elif BASE_PATH.name != "oracle_mnc_project" and (BASE_PATH / "oracle_mnc_project").exists():
    BASE_PATH = BASE_PATH / "oracle_mnc_project"

ACCESS_OUTPUT_PATH = BASE_PATH / "notebooks" / "access" / "OUTPUT"
PUBLIC_PATH = ACCESS_OUTPUT_PATH / "public_access_index_25km"
H3_PATH = ACCESS_OUTPUT_PATH / "h3sfca"
SENSITIVITY_PATH = ACCESS_OUTPUT_PATH / "h3sfca_sensitivity"
COMPARE_PATH = ACCESS_OUTPUT_PATH / "compare_index"

COMPARE_PATH.mkdir(parents=True, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("PUBLIC_PATH:", PUBLIC_PATH)
print("H3_PATH:", H3_PATH)
print("SENSITIVITY_PATH:", SENSITIVITY_PATH)
print("COMPARE_PATH:", COMPARE_PATH)


## 2. 이용건수 타깃 테이블

- 2025년 서울 구·중분류 이용건수를 기준값으로 사용함.
- H3SFCA 검증 테이블에서 중복 제거 후 타깃만 추출함.
- 음악·체육용품은 선호모형 제외 기준과 동일하게 제외됨.


In [ ]:
# 2025년 이용건수 타깃
usage_model = pd.read_csv(
    SENSITIVITY_PATH / "h3sfca_usage_regression_2025_model_data.csv",
    encoding="utf-8-sig"
)

usage = (
    usage_model[["시군구", "중분류", "이용건수"]]
    .drop_duplicates()
    .copy()
)

usage["이용건수"] = pd.to_numeric(usage["이용건수"], errors="coerce").fillna(0)

category_list = sorted(usage["중분류"].unique())

print("이용건수 타깃 구조:", usage.shape)
print("시군구 수:", usage["시군구"].nunique())
print("중분류 수:", usage["중분류"].nunique())
print("중분류:", category_list)
print("이용건수 결측:", usage["이용건수"].isna().sum())

display(
    usage
    .groupby("중분류", as_index=False)["이용건수"]
    .sum()
    .sort_values("이용건수", ascending=False)
)


## 3. 정부식 접근성 지표 불러오기

- 정부식 최근접 접근성은 대상자 가중평균 접근거리를 역거리로 변환함.
- 서비스권역비율은 10km 이내 접근 가능한 문화누리대상자 비율을 사용함.
- 우리반경 버전은 이전에 추가한 도보·대중교통 기준의 보조 비교지표로 함께 확인함.


In [ ]:
# 정부식 접근성 지표
gov_nearest = pd.read_csv(
    PUBLIC_PATH / "공공기관식_최근접접근성_서울시군구_중분류별.csv",
    encoding="utf-8-sig"
)

gov_service = pd.read_csv(
    PUBLIC_PATH / "공공기관식_서비스권역인구비율_서울시군구_중분류별.csv",
    encoding="utf-8-sig"
)

our_radius = pd.read_csv(
    PUBLIC_PATH / "공공기관식_우리반경_최근접접근성_서울시군구_중분류별.csv",
    encoding="utf-8-sig"
)

indicator_list = []

temp = gov_nearest[["시군구", "중분류", "문화누리대상자_가중평균_접근거리_m"]].copy()
temp["지표명"] = "정부10km_최근접접근성"
temp["지표값"] = 1 / (1 + pd.to_numeric(temp["문화누리대상자_가중평균_접근거리_m"], errors="coerce"))
temp["원지표값"] = temp["문화누리대상자_가중평균_접근거리_m"]
temp["지표해석"] = "역거리_높을수록접근성높음"
indicator_list.append(temp[["시군구", "중분류", "지표명", "지표값", "원지표값", "지표해석"]])

temp = gov_service[["시군구", "중분류", "문화누리대상자_서비스권역비율"]].copy()
temp["지표명"] = "정부10km_서비스권역비율"
temp["지표값"] = pd.to_numeric(temp["문화누리대상자_서비스권역비율"], errors="coerce")
temp["원지표값"] = temp["지표값"]
temp["지표해석"] = "비율_높을수록접근성높음"
indicator_list.append(temp[["시군구", "중분류", "지표명", "지표값", "원지표값", "지표해석"]])

temp = our_radius[["시군구", "중분류", "우리기준_대상자_가중평균_접근비용"]].copy()
temp["지표명"] = "우리반경_최근접접근성"
temp["지표값"] = 1 / (1 + pd.to_numeric(temp["우리기준_대상자_가중평균_접근비용"], errors="coerce"))
temp["원지표값"] = temp["우리기준_대상자_가중평균_접근비용"]
temp["지표해석"] = "역비용_높을수록접근성높음"
indicator_list.append(temp[["시군구", "중분류", "지표명", "지표값", "원지표값", "지표해석"]])

temp = our_radius[["시군구", "중분류", "우리기준_서비스권역비율"]].copy()
temp["지표명"] = "우리반경_서비스권역비율"
temp["지표값"] = pd.to_numeric(temp["우리기준_서비스권역비율"], errors="coerce")
temp["원지표값"] = temp["지표값"]
temp["지표해석"] = "비율_높을수록접근성높음"
indicator_list.append(temp[["시군구", "중분류", "지표명", "지표값", "원지표값", "지표해석"]])

government_indicator = pd.concat(indicator_list, ignore_index=True)
government_indicator = government_indicator[government_indicator["중분류"].isin(category_list)].copy()

print("정부식 접근성 지표 구조:", government_indicator.shape)
print("정부식 지표 종류")
print(government_indicator["지표명"].value_counts())
display(government_indicator.head())


## 4. SFCA 계열 접근성 지표 불러오기

- 선호 미반영 SFCA는 격자 접근성을 구·중분류 단위로 대상자 가중평균함.
- 선호 반영 H3SFCA는 기존 민감도 검증용 구·중분류 테이블을 사용함.
- 선호 반영 H3SFCA는 9개 시나리오를 모두 비교함.


In [ ]:
def weighted_mean(group, value_col, weight_col):
    value = pd.to_numeric(group[value_col], errors="coerce").fillna(0)
    weight = pd.to_numeric(group[weight_col], errors="coerce").fillna(0)
    
    if weight.sum() > 0:
        return np.average(value, weights=weight)
    return value.mean()


# 선호 미반영 SFCA
sfca_grid = pd.read_csv(
    H3_PATH / "sfca_no_preference_격자_중분류_접근성.csv",
    encoding="utf-8-sig",
    usecols=["시군구", "중분류", "접근성지수", "문화누리대상자_추정_인구수"]
)

sfca_grid = sfca_grid[sfca_grid["중분류"].isin(category_list)].copy()

sfca_no_preference = (
    sfca_grid
    .groupby(["시군구", "중분류"], as_index=False)
    .apply(
        lambda x: pd.Series({
            "지표값": weighted_mean(x, "접근성지수", "문화누리대상자_추정_인구수"),
            "원지표값": weighted_mean(x, "접근성지수", "문화누리대상자_추정_인구수")
        }),
        include_groups=False
    )
)

sfca_no_preference["지표명"] = "SFCA_선호미반영"
sfca_no_preference["지표해석"] = "공급수요비_높을수록접근성높음"
sfca_no_preference = sfca_no_preference[["시군구", "중분류", "지표명", "지표값", "원지표값", "지표해석"]]

print("선호 미반영 SFCA 구조:", sfca_no_preference.shape)
display(sfca_no_preference.head())


# 선호 반영 H3SFCA
h3sfca_indicator = (
    usage_model[["시군구", "중분류", "scenario", "H3SFCA_접근성"]]
    .drop_duplicates()
    .copy()
)

h3sfca_indicator["지표명"] = "H3SFCA_선호반영_" + h3sfca_indicator["scenario"].astype(str)
h3sfca_indicator["지표값"] = pd.to_numeric(h3sfca_indicator["H3SFCA_접근성"], errors="coerce")
h3sfca_indicator["원지표값"] = h3sfca_indicator["지표값"]
h3sfca_indicator["지표해석"] = "선호반영_공급수요비_높을수록접근성높음"
h3sfca_indicator = h3sfca_indicator[["시군구", "중분류", "지표명", "지표값", "원지표값", "지표해석"]]

print("선호 반영 H3SFCA 구조:", h3sfca_indicator.shape)
print("H3SFCA 시나리오 수:", h3sfca_indicator["지표명"].nunique())
display(h3sfca_indicator.head())


## 5. 지표 통합 및 품질 점검

- 모든 지표를 `시군구 × 중분류 × 지표명` 구조로 통합함.
- 이용건수와 결합되지 않는 레코드가 있는지 확인함.
- 상관분석은 원 이용건수 기준으로 수행함.


In [ ]:
# 접근성 지표 통합
access_indicator = pd.concat(
    [
        government_indicator,
        sfca_no_preference,
        h3sfca_indicator
    ],
    ignore_index=True
)

compare_data = access_indicator.merge(
    usage,
    on=["시군구", "중분류"],
    how="left"
)

print("통합 접근성 지표 구조:", access_indicator.shape)
print("비교 데이터 구조:", compare_data.shape)
print("지표 수:", compare_data["지표명"].nunique())
print("시군구 수:", compare_data["시군구"].nunique())
print("중분류 수:", compare_data["중분류"].nunique())
print("이용건수 결측:", compare_data["이용건수"].isna().sum())
print("지표값 결측:", compare_data["지표값"].isna().sum())

display(
    compare_data
    .groupby("지표명", as_index=False)
    .agg(
        레코드수=("지표값", "size"),
        지표값결측=("지표값", lambda x: x.isna().sum()),
        이용건수결측=("이용건수", lambda x: x.isna().sum()),
        지표값표준편차=("지표값", "std"),
    )
    .sort_values("지표명")
)


## 6. 이용건수 상관분석

- Pearson: 접근성 값과 이용건수의 선형 상관을 확인함.
- Spearman: 접근성 순위와 이용건수 순위의 상관을 확인함.
- 전체 상관과 중분류별 상관을 함께 산출함.
- 중분류별 상관 평균은 분류 간 이용규모 차이를 줄여 해석하기 위한 보조 결과임.


In [ ]:
def safe_corr(df, x_col, y_col, method):
    temp = df[[x_col, y_col]].dropna()
    
    if len(temp) < 3:
        return np.nan
    if temp[x_col].nunique() < 2 or temp[y_col].nunique() < 2:
        return np.nan
    
    return temp[x_col].corr(temp[y_col], method=method)


# 전체 상관
overall_corr_list = []

for indicator_name, temp in compare_data.groupby("지표명"):
    overall_corr_list.append({
        "지표명": indicator_name,
        "n": len(temp),
        "Pearson": safe_corr(temp, "지표값", "이용건수", "pearson"),
        "Spearman": safe_corr(temp, "지표값", "이용건수", "spearman"),
        "지표값_평균": temp["지표값"].mean(),
        "지표값_표준편차": temp["지표값"].std(),
        "이용건수_평균": temp["이용건수"].mean(),
    })

overall_corr = pd.DataFrame(overall_corr_list)


# 중분류별 상관
category_corr_list = []

for (indicator_name, category), temp in compare_data.groupby(["지표명", "중분류"]):
    category_corr_list.append({
        "지표명": indicator_name,
        "중분류": category,
        "n": len(temp),
        "Pearson": safe_corr(temp, "지표값", "이용건수", "pearson"),
        "Spearman": safe_corr(temp, "지표값", "이용건수", "spearman"),
        "지표값_평균": temp["지표값"].mean(),
        "지표값_표준편차": temp["지표값"].std(),
        "이용건수_합": temp["이용건수"].sum(),
    })

category_corr = pd.DataFrame(category_corr_list)


# 중분류별 상관 요약
category_corr_summary = (
    category_corr
    .groupby("지표명", as_index=False)
    .agg(
        중분류수=("중분류", "nunique"),
        Pearson_평균=("Pearson", "mean"),
        Pearson_중앙값=("Pearson", "median"),
        Pearson_양수분류수=("Pearson", lambda x: (x > 0).sum()),
        Spearman_평균=("Spearman", "mean"),
        Spearman_중앙값=("Spearman", "median"),
        Spearman_양수분류수=("Spearman", lambda x: (x > 0).sum()),
    )
)

overall_corr_path = COMPARE_PATH / "compare_index_2025_overall_correlation.csv"
category_corr_path = COMPARE_PATH / "compare_index_2025_category_correlation.csv"
category_summary_path = COMPARE_PATH / "compare_index_2025_category_summary.csv"

overall_corr.to_csv(overall_corr_path, index=False, encoding="utf-8-sig")
category_corr.to_csv(category_corr_path, index=False, encoding="utf-8-sig")
category_corr_summary.to_csv(category_summary_path, index=False, encoding="utf-8-sig")

print("전체 상관 저장:", overall_corr_path)
print("중분류별 상관 저장:", category_corr_path)
print("중분류별 상관 요약 저장:", category_summary_path)

print("\n전체 상관: Spearman 높은 순")
display(overall_corr.sort_values("Spearman", ascending=False))

print("\n중분류별 상관 평균: Spearman 평균 높은 순")
display(category_corr_summary.sort_values("Spearman_평균", ascending=False))

print("\n중분류별 상세 상관")
display(category_corr.sort_values(["지표명", "중분류"]))


## 7. 해석 방향

- 전체 상관은 중분류 간 이용건수 규모 차이의 영향을 받을 수 있음.
- 중분류별 상관은 같은 분류 안에서 구별 접근성과 이용건수 순위가 같이 움직이는지 확인하는 결과임.
- 서비스권역비율은 값이 100%에 가까우면 변동성이 작아 상관분석에서 불리할 수 있음.
- 최근접 접근성은 역거리·역비용으로 변환했으므로 값이 높을수록 접근성이 높음.


## 8. 주요 결과 요약

- 전체 상관 기준에서는 정부10km 최근접 접근성이 가장 높게 나타남.
- 중분류별 상관 평균 기준에서는 정부10km 서비스권역비율이 가장 높게 나타남.
- 선호 미반영 SFCA와 선호 반영 H3SFCA는 전체적으로 이용건수와 약한 상관 또는 음의 상관을 보임.
- H3SFCA는 체육시설에서는 양의 상관이 높지만, 공연·도서·미술·영상에서는 음의 상관이 반복됨.
- 우리반경 최근접 접근성은 도봉구 스포츠관람에서 접근 가능 대상자가 0명이라 지표값 결측 1건이 발생함.

### 전체 상관 상위 5개
| 지표명 | Pearson | Spearman |
|---|---:|---:|
| 정부10km_최근접접근성 | 0.109 | 0.350 |
| 정부10km_서비스권역비율 | 0.101 | 0.235 |
| H3SFCA_선호반영_none_lambda_1.0 | -0.230 | 0.039 |
| H3SFCA_선호반영_piecewise_lambda_1.0 | -0.225 | 0.039 |
| H3SFCA_선호반영_none_lambda_1.2 | -0.230 | 0.039 |

### 중분류별 상관 평균 상위 5개
| 지표명 | Pearson 평균 | Spearman 평균 | Spearman 양수 분류 수 |
|---|---:|---:|---:|
| 정부10km_서비스권역비율 | 0.157 | 0.156 | 7/8 |
| 우리반경_서비스권역비율 | 0.069 | 0.105 | 5/8 |
| 우리반경_최근접접근성 | 0.095 | 0.081 | 5/8 |
| 정부10km_최근접접근성 | 0.128 | 0.053 | 5/8 |
| H3SFCA_선호반영_none_lambda_1.2 | -0.089 | -0.110 | 2/8 |

### 해석
- 이용건수 상관만 놓고 보면 정부식 지표가 SFCA 계열보다 더 우호적으로 나옴.
- 다만 정부식 서비스권역비율은 대부분 100%에 가까워 정책 취약지역 식별력은 제한적일 수 있음.
- SFCA/H3SFCA가 불리하게 나온 이유는 이용건수가 접근성뿐 아니라 분류별 소비규모, 가격, 이용행태, 특정 대형시설 효과의 영향을 크게 받기 때문으로 해석됨.


## 9. 수요 보정 이용지표 상관분석

- H3SFCA 격자 접근성은 구·중분류 단위로 선호수요 가중평균함.
- 선호 미반영 SFCA는 문화누리대상자 추정인구로 가중평균함.
- 이용건수는 원 이용건수, 대상자당 이용건수, 선호수요당 이용건수로 나누어 비교함.
- 선호수요당 이용건수의 분모는 기본 시나리오(`piecewise_lambda_1.2`)의 선호수요로 고정함.


In [ ]:
# 수요 보정 이용지표 상관분석
def safe_corr(df, x_col, y_col, method):
    temp = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(temp) < 3:
        return np.nan
    if temp[x_col].nunique() < 2 or temp[y_col].nunique() < 2:
        return np.nan
    
    return temp[x_col].corr(temp[y_col], method=method)


def weighted_avg(value, weight):
    value = pd.to_numeric(value, errors="coerce").fillna(0)
    weight = pd.to_numeric(weight, errors="coerce").fillna(0)
    
    if weight.sum() > 0:
        return np.average(value, weights=weight)
    return value.mean()


# 이용건수 및 대상자 수
usage = (
    usage_model[["시군구", "중분류", "이용건수", "구별_문화누리대상자추정인구"]]
    .drop_duplicates()
    .copy()
)

usage["이용건수"] = pd.to_numeric(usage["이용건수"], errors="coerce").fillna(0)
usage["구별_문화누리대상자추정인구"] = pd.to_numeric(
    usage["구별_문화누리대상자추정인구"],
    errors="coerce"
).fillna(0)


# H3SFCA: 격자 접근성을 구·중분류 단위로 선호수요 가중평균
h3_grid = pd.read_parquet(
    H3_PATH / "h3sfca_preference_sensitivity_격자_중분류_접근성.parquet",
    columns=[
        "시군구",
        "중분류",
        "접근성지수",
        "문화누리대상자_추정_인구수",
        "선호수요",
        "거리감쇠방식",
        "문화누리대상자_수요가중치",
    ],
)

h3_grid = h3_grid[h3_grid["중분류"].isin(category_list)].copy()
h3_grid["scenario"] = (
    h3_grid["거리감쇠방식"].astype(str)
    + "_lambda_"
    + h3_grid["문화누리대상자_수요가중치"].astype(float).astype(str)
)

h3_group_list = []

for keys, temp in h3_grid.groupby(["시군구", "중분류", "scenario"], observed=True):
    gu, category, scenario = keys
    preference_weight = temp["선호수요"].clip(lower=0)
    fallback_weight = temp["문화누리대상자_추정_인구수"].clip(lower=0)
    final_weight = preference_weight if preference_weight.sum() > 0 else fallback_weight
    
    h3_group_list.append({
        "시군구": gu,
        "중분류": category,
        "지표명": "H3SFCA_선호반영_" + scenario,
        "지표값": weighted_avg(temp["접근성지수"], final_weight),
        "구별_선호수요합": preference_weight.sum(),
    })

h3_indicator = pd.DataFrame(h3_group_list)

preference_demand_check = (
    h3_indicator
    .groupby(["시군구", "중분류"], as_index=False)["구별_선호수요합"]
    .agg(["min", "max"])
    .reset_index()
)
preference_demand_check["선호수요_시나리오차이"] = (
    preference_demand_check["max"] - preference_demand_check["min"]
)


# 선호수요당 이용건수의 고정 분모: 기본 시나리오 선호수요
h3_default = pd.read_csv(
    H3_PATH / "h3sfca_격자_중분류_접근성.csv",
    encoding="utf-8-sig",
    usecols=["시군구", "중분류", "선호수요"]
)

h3_default = h3_default[h3_default["중분류"].isin(category_list)].copy()

preference_demand = (
    h3_default
    .groupby(["시군구", "중분류"], as_index=False)["선호수요"]
    .sum()
    .rename(columns={"선호수요": "기본시나리오_구별_선호수요합"})
)


# SFCA: 격자 접근성을 구·중분류 단위로 대상자 가중평균
sfca_grid = pd.read_csv(
    H3_PATH / "sfca_no_preference_격자_중분류_접근성.csv",
    encoding="utf-8-sig",
    usecols=["시군구", "중분류", "접근성지수", "문화누리대상자_추정_인구수"]
)

sfca_grid = sfca_grid[sfca_grid["중분류"].isin(category_list)].copy()

sfca_group_list = []
for keys, temp in sfca_grid.groupby(["시군구", "중분류"], observed=True):
    gu, category = keys
    weight = temp["문화누리대상자_추정_인구수"].clip(lower=0)
    sfca_group_list.append({
        "시군구": gu,
        "중분류": category,
        "지표명": "SFCA_선호미반영",
        "지표값": weighted_avg(temp["접근성지수"], weight),
    })

sfca_indicator = pd.DataFrame(sfca_group_list)


# 정부식 두 지표
gov_nearest = pd.read_csv(
    PUBLIC_PATH / "공공기관식_최근접접근성_서울시군구_중분류별.csv",
    encoding="utf-8-sig"
)
gov_service = pd.read_csv(
    PUBLIC_PATH / "공공기관식_서비스권역인구비율_서울시군구_중분류별.csv",
    encoding="utf-8-sig"
)

gov_list = []

temp = gov_nearest[["시군구", "중분류", "문화누리대상자_가중평균_접근거리_m"]].copy()
temp = temp[temp["중분류"].isin(category_list)].copy()
temp["지표명"] = "정부10km_최근접접근성"
temp["지표값"] = 1 / (1 + pd.to_numeric(temp["문화누리대상자_가중평균_접근거리_m"], errors="coerce"))
gov_list.append(temp[["시군구", "중분류", "지표명", "지표값"]])

temp = gov_service[["시군구", "중분류", "문화누리대상자_서비스권역비율"]].copy()
temp = temp[temp["중분류"].isin(category_list)].copy()
temp["지표명"] = "정부10km_서비스권역비율"
temp["지표값"] = pd.to_numeric(temp["문화누리대상자_서비스권역비율"], errors="coerce")
gov_list.append(temp[["시군구", "중분류", "지표명", "지표값"]])

gov_indicator = pd.concat(gov_list, ignore_index=True)


# 통합 및 수요 보정 이용지표 생성
indicator = pd.concat(
    [
        gov_indicator,
        sfca_indicator,
        h3_indicator[["시군구", "중분류", "지표명", "지표값"]],
    ],
    ignore_index=True
)

compare = (
    indicator
    .merge(usage, on=["시군구", "중분류"], how="left")
    .merge(preference_demand, on=["시군구", "중분류"], how="left")
)

compare["대상자당_이용건수"] = np.where(
    compare["구별_문화누리대상자추정인구"] > 0,
    compare["이용건수"] / compare["구별_문화누리대상자추정인구"] * 1000,
    np.nan
)

compare["선호수요당_이용건수"] = np.where(
    compare["기본시나리오_구별_선호수요합"] > 0,
    compare["이용건수"] / compare["기본시나리오_구별_선호수요합"] * 1000,
    np.nan
)


# 상관분석
target_cols = ["이용건수", "대상자당_이용건수", "선호수요당_이용건수"]

overall_rows = []
category_rows = []

for target_col in target_cols:
    for indicator_name, temp in compare.groupby("지표명", observed=True):
        overall_rows.append({
            "이용지표": target_col,
            "지표명": indicator_name,
            "n": len(temp),
            "Pearson": safe_corr(temp, "지표값", target_col, "pearson"),
            "Spearman": safe_corr(temp, "지표값", target_col, "spearman"),
        })
    
    for (indicator_name, category), temp in compare.groupby(["지표명", "중분류"], observed=True):
        category_rows.append({
            "이용지표": target_col,
            "지표명": indicator_name,
            "중분류": category,
            "n": len(temp),
            "Pearson": safe_corr(temp, "지표값", target_col, "pearson"),
            "Spearman": safe_corr(temp, "지표값", target_col, "spearman"),
        })

overall_corr = pd.DataFrame(overall_rows)
category_corr = pd.DataFrame(category_rows)

category_summary = (
    category_corr
    .groupby(["이용지표", "지표명"], as_index=False)
    .agg(
        Pearson_평균=("Pearson", "mean"),
        Pearson_중앙값=("Pearson", "median"),
        Pearson_양수분류수=("Pearson", lambda x: (x > 0).sum()),
        Spearman_평균=("Spearman", "mean"),
        Spearman_중앙값=("Spearman", "median"),
        Spearman_양수분류수=("Spearman", lambda x: (x > 0).sum()),
    )
)

overall_corr.to_csv(
    COMPARE_PATH / "compare_index_2025_demand_adjusted_overall_correlation.csv",
    index=False,
    encoding="utf-8-sig"
)
category_corr.to_csv(
    COMPARE_PATH / "compare_index_2025_demand_adjusted_category_correlation.csv",
    index=False,
    encoding="utf-8-sig"
)
category_summary.to_csv(
    COMPARE_PATH / "compare_index_2025_demand_adjusted_category_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("비교 데이터 구조:", compare.shape)
print("지표값 결측:", compare["지표값"].isna().sum())
print("선호수요당 이용건수 결측:", compare["선호수요당_이용건수"].isna().sum())
print("선호수요 시나리오별 차이 최대:", preference_demand_check["선호수요_시나리오차이"].abs().max())

display(
    overall_corr[overall_corr["이용지표"] == "대상자당_이용건수"]
    .sort_values("Spearman", ascending=False)
)

display(
    category_summary[category_summary["이용지표"] == "대상자당_이용건수"]
    .sort_values("Spearman_평균", ascending=False)
)


### 9-1. 수요 보정 분석 결과

#### 대상자당 이용건수 기준: 전체 상관 상위 6개
| 지표명 | Pearson | Spearman |
|---|---:|---:|
| 정부10km_최근접접근성 | 0.119 | 0.375 |
| 정부10km_서비스권역비율 | 0.115 | 0.194 |
| H3SFCA_선호반영_none_lambda_1.0 | -0.232 | 0.112 |
| H3SFCA_선호반영_none_lambda_1.2 | -0.232 | 0.112 |
| H3SFCA_선호반영_gaussian_lambda_1.2 | -0.221 | 0.112 |
| H3SFCA_선호반영_none_lambda_1.5 | -0.232 | 0.112 |

#### 대상자당 이용건수 기준: 중분류별 상관 평균 상위 6개
| 지표명 | Pearson 평균 | Spearman 평균 | Spearman 양수 분류 수 |
|---|---:|---:|---:|
| 정부10km_최근접접근성 | 0.113 | 0.084 | 4/8 |
| SFCA_선호미반영 | 0.110 | 0.077 | 4/8 |
| H3SFCA_선호반영_none_lambda_1.2 | 0.072 | 0.062 | 4/8 |
| H3SFCA_선호반영_none_lambda_1.0 | 0.072 | 0.059 | 4/8 |
| H3SFCA_선호반영_none_lambda_1.5 | 0.072 | 0.056 | 4/8 |
| H3SFCA_선호반영_piecewise_lambda_1.2 | 0.062 | 0.043 | 3/8 |

#### 선호수요당 이용건수 기준: 중분류별 상관 평균 상위 6개
| 지표명 | Pearson 평균 | Spearman 평균 | Spearman 양수 분류 수 |
|---|---:|---:|---:|
| 정부10km_최근접접근성 | 0.277 | 0.252 | 6/8 |
| 정부10km_서비스권역비율 | 0.192 | 0.250 | 7/8 |
| SFCA_선호미반영 | 0.009 | 0.038 | 4/8 |
| H3SFCA_선호반영_none_lambda_1.0 | 0.031 | -0.001 | 3/8 |
| H3SFCA_선호반영_none_lambda_1.2 | 0.028 | -0.005 | 3/8 |
| H3SFCA_선호반영_piecewise_lambda_1.0 | 0.014 | -0.011 | 3/8 |

#### 해석
- 구별 단순 합산 문제를 피하기 위해 접근성은 수요가중평균으로 집계함.
- 이용건수를 대상자 기준으로 보정하면 H3SFCA 계열의 Spearman 상관은 기존보다 일부 개선됨.
- 다만 정부10km 최근접 접근성이 여전히 전체 상관에서 가장 높음.
- 선호수요당 이용건수 기준에서는 정부식 지표가 더 높고, H3SFCA 계열은 평균적으로 0에 가깝거나 음수임.
- 따라서 기존 음의 상관은 회귀모형만의 문제가 아니라, 구 단위 접근성과 이용지표의 관계 자체가 약하거나 분류별로 다르게 나타난 결과로 해석함.
